In [1]:
## Create a vector of the required packages for this analysis
req_packages <- c("tidyverse")

## load the packages, quietly
invisible(suppressWarnings(suppressMessages(
    lapply(req_packages, require, character.only = TRUE)
)))

## Load in GO file

In [2]:
## load in file
pse_id_raw <- read_tsv("results/Trinotate_report.tsv.gene_ontology",col_names = FALSE)
colnames(pse_id_raw) <- c("gene", "go_terms")

## reformat file
pse_id <- pse_id_raw %>%
    mutate(gene = str_replace(gene, "gene-", "")) %>%
    separate_longer_delim(go_terms, delim = ",")
head(pse_id)

Rows: 11388 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): X1, X2

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


gene,go_terms
<chr>,<chr>
HCS66_mgp01,GO:0003674
HCS66_mgp01,GO:0003824
HCS66_mgp01,GO:0004129
HCS66_mgp01,GO:0005215
HCS66_mgp01,GO:0005488
HCS66_mgp01,GO:0005507


In [3]:
## create function for converting GO IDs to GO terms
goIdToTerm <- function(x, names = TRUE, keepNA = TRUE) {
    ans <- rep(NA_character_, length(x))
    names(ans) <- x
    ids <- AnnotationDbi::GOID(GO.db::GOTERM)
    i <- match(x, ids)
    k <- which(!is.na(i))
    res <- AnnotationDbi::Term(GO.db::GOTERM[i[k]])
    ans[k] <- res
    if (!keepNA) ans[is.na(ans)] <- names(ans[is.na(ans)])
    if (!names) names(ans) <- NULL
    return(ans)
}

In [4]:
## get list of GO IDs
go_id <- pse_id %>%
    pull("go_terms")

## convert GO IDs to GO terms
go_term <- goIdToTerm(go_id, names = FALSE)

## add GO terms to dataframe
pse_id$term <- go_term

## clean up data frame
pse_go <- pse_id %>%
    select(-go_terms) %>%
    na.omit()

head(pse_go)

gene,term
<chr>,<chr>
HCS66_mgp01,molecular_function
HCS66_mgp01,catalytic activity
HCS66_mgp01,cytochrome-c oxidase activity
HCS66_mgp01,transporter activity
HCS66_mgp01,binding
HCS66_mgp01,copper ion binding


In [5]:
## label GO terms by category

## create empty data frame
pse_go_label <- data.frame(gene = NA, term = NA, category = NA)

## get lists to iterate through
    ## GO term categories
categories <- c("molecular_function", "biological_process", "cellular_component")
    ## genes
genes <- unique(pse_go$gene)

for (i in genes) {
    
    ## filter for only one gene
    pse_go_select <- pse_go %>%
        filter(gene == i)

    for (j in 1:(length(categories) - 1)) {
            
            ## get the name of the corresponding category
            category1 <- categories[j] 

            ## check to see if category is in dataframe
            row_category_1 <- which(pse_go_select$term == category1)

        if (length(row_category_1) != 0) {

            ## get the range of the first category, not including category name
            row_category_1 <- which(pse_go_select$term == category1) + 1

            for (k in (j + 1):length(categories)) {    
                
                ## get the name of the second category
                category2 <- categories[k]

                ## check to see if second category is in dataframe
                row_category_2 <- which(pse_go_select$term == category2)       
                
                if (length(row_category_2) != 0) {
                    
                    ## get the range of the second category, not including category name
                    row_category_2 <- which(pse_go_select$term == category2) - 1

                    ## filter for only rows of specific category, and add column for category label
                    pse_go_slice <- pse_go_select %>%
                        slice(row_category_1:row_category_2) %>%
                        mutate(category = category1)

                    ## add rows to dataframe
                    pse_go_label <- rbind(pse_go_label, pse_go_slice)

                } 
                else {
                    
                    ## try the next category
                    k <- k+1
                    category2 <- categories[k]

                    if (length(category2) != 0) {
                        
                        ## check if category exists
                        row_category_2 <- which(pse_go_select$term == category2)

                        if (length(row_category_2) != 0) {
                            
                            row_category_2 <- which(pse_go_select$term == category2) - 1
                            pse_go_slice <- pse_go_select %>%
                                slice(row_category_1:row_category_2) %>%
                                mutate(category = category1)
                            pse_go_label <- rbind(pse_go_label, pse_go_slice)
                        
                        }

                    } else {
                        
                        ## run until the end
                        row_category_2 <- nrow(pse_go_select)

                        ## filter for only rows of specific category, and add column for category label
                        pse_go_slice <- pse_go_select %>%
                            slice(row_category_1:row_category_2) %>%
                            mutate(category = category1)

                        ## add rows to dataframe
                        pse_go_label <- rbind(pse_go_label, pse_go_slice)

                    }

                }
            
            }
        }
    }
    
    ## repeat above steps for last category
    j <- length(categories)
    category1 <- categories[j]

    ## check to see if category is in dataframe
    row_category_1 <- which(pse_go_select$term == category1)

    if (length(row_category_1) != 0) {  

        ## get the range of each category, not including category name
        row_category_1 <- which(pse_go_select$term == category1) + 1
        row_category_2 <- nrow(pse_go_select)

        ## filter for only rows of specific category, and add column for category label
        pse_go_slice <- pse_go_select %>%
            slice(row_category_1:row_category_2) %>%
            mutate(category = category1)

        ## add rows to dataframe
        pse_go_label <- rbind(pse_go_label, pse_go_slice)
    
    }
}

## add molecular function in case any GO terms are above it in order
pse_go_label <- pse_go_label %>%
    na.omit() %>%
    full_join(pse_go, by = c("gene", "term")) %>%
    mutate(category = case_when(is.na(category) ~ "molecular_function",
                                TRUE ~ category))

write.csv(pse_go_label, "results/dpse_goterms.csv")